# Implementing GANs — Hands-on Training of a Simple GAN for Image Generation
---

## Learning objectives

By the end of this lab, you will be able to:

- Explain the **minimax game** between a **generator** $G$ and **discriminator** $D$, and interpret the **value function** used in the original GAN formulation.
- Implement a **vanilla GAN** in PyTorch using **MLPs** on **MNIST** (flattened images).
- Train with **BCE loss**, **Adam**, and **proper label tensors** for real vs. fake samples.
- Visualize **generated samples** and **loss curves**, and connect trends to **training stability**.
- Apply practical **stabilization ideas** (e.g., weight initialization, label smoothing) and **experiment** with architecture and data.

---

## GAN theory (compact refresher)

Generative Adversarial Networks (Goodfellow et al., 2014) frame generation as a **two-player game**:

- **Generator** $G(z)$ maps random noise $z \sim p_z$ (e.g., Gaussian) to data-like samples (here, fake MNIST digits).
- **Discriminator** $D(x)$ outputs a **probability** that input $x$ is **real** (from the dataset) rather than **fake** (from $G$).

**Value function (vanilla GAN, conceptual):**

$$\min_G \max_D \; V(D,G) = \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]$$

- $D$ tries to **maximize** $\log D(x)$ on real data and **minimize** $D(G(z))$ on fakes (equivalently maximize $\log(1-D(G(z)))$).
- $G$ tries to **fool** $D$, minimizing $\log(1-D(G(z)))$, often implemented by minimizing $-\log(D(G(z)))$ (**non-saturating** generator objective) for stronger gradients when $D$ is confident.

In this lab we use **binary cross-entropy** (`BCELoss`) with **target 1** for real and **target 0** for fake when training $D$; for $G$ we use fake samples with target **1** ("want discriminator to think fake is real").

---

## Prerequisites

- Comfortable with **PyTorch** basics: `nn.Module`, `optim`, tensors, `device` placement.
- Understanding of **binary classification** and **sigmoid + BCE**.
- Familiarity with **MNIST** / image tensors (optional: convolutional GANs are *not* required here — we use **MLPs** on **flattened** images).

## Environment

- **Google Colab** or local **Jupyter** with **PyTorch** + **torchvision** + **matplotlib** + **numpy**.
- Enable **GPU** in Colab (**Runtime → Change runtime type → GPU**) for faster training; the notebook runs on **CPU** as well, only slower.

> Run cells **in order** the first time through.


---

# Tutorial: Step-by-Step Implementation of a Vanilla GAN

This section walks you from **data** → **models** → **training** → **visualization**. Read the markdown between code cells: it explains *what* each block does and *why*.


### Setup: imports, reproducibility, and device

We fix **random seeds** for reproducibility (results may still vary slightly across hardware). **`torch.cuda.is_available()`** selects **GPU** when present.


In [ ]:
# Cell: imports, seed, device
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


**What you should see:** a one-line print with `cuda` (GPU) or `cpu`. All later tensors and models will be moved to this `device`.


### Data: MNIST, flattened and scaled to [-1, 1]

- **`ToTensor()`** gives floats in **[0, 1]**.
- We **flatten** to 784-D vectors with **`Lambda`**.
- **GANs** often use **Tanh** outputs; matching the data range to **[-1, 1]** works well: `x' = 2x - 1`.

**Batch size** is **128** as specified.


In [ ]:
batch_size = 128

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),  # flatten to 784
    transforms.Lambda(lambda x: 2.0 * x - 1.0),  # [0,1] -> [-1,1]
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform,
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

print(f"MNIST train samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")


**Takeaway:** each batch has shape **`(128, 784)`** after flattening. Discriminator sees **vectors**; this is a **vanilla MLP GAN**, not DCGAN.


### Models: MLP Generator and Discriminator

**Generator:** noise dimension **`nz = 100`**. Hidden layers use **LeakyReLU** (small slope for negative inputs helps gradients). Output is **784** with **Tanh** so outputs lie in **[-1, 1]** (matching images).

**Discriminator:** maps **784 → 1** with **Sigmoid** (probability "real").

**Weight initialization:** small Gaussian noise for weights; zeros for biases — a common starting point (you may compare with default init in the exercises).


In [ ]:
nz = 100  # noise dimension

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Linear') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self, nz: int = 100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nz, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 784),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


G = Generator(nz).to(device)
D = Discriminator().to(device)
G.apply(weights_init)
D.apply(weights_init)

print(G)
print(D)


**Sanity check:** parameter counts should be modest (MLPs on 784-D). If shapes mismatch later, re-check **784** and **`nz`**.


### Loss and optimizers

- **`BCELoss`** matches **sigmoid** outputs.
- **Adam** with **`lr=2e-4`**, **`betas=(0.5, 0.999)`** — **beta1 = 0.5** is a common heuristic for GANs (less momentum than default 0.9).

**Labels:** real batch → **1**, fake batch → **0** when training **D**; for **G**, we pass **fakes** with target **1**.


In [ ]:
criterion = nn.BCELoss()
lr = 0.0002
betas = (0.5, 0.999)

optimizerD = optim.Adam(D.parameters(), lr=lr, betas=betas)
optimizerG = optim.Adam(G.parameters(), lr=lr, betas=betas)


Separate optimizers let us update **D** and **G** on **different** objectives while sharing the same hyperparameters unless you choose otherwise in the exercises.


### Visualization helpers

We will plot **64** generated digits in an **8×8** grid and track **mean** generator/discriminator loss per epoch.


In [ ]:
def denormalize_to_01(x: torch.Tensor) -> torch.Tensor:
    """Map from [-1, 1] back to [0, 1] for plotting."""
    return (x + 1.0) / 2.0


The training loop below uses **`torch.no_grad()`** when rendering snapshots so we **do not** build computation graphs for visualization-only forwards.


### Training loop (50 epochs)

Each epoch:

1. **Update D:** classify real vs fake.
2. **Update G:** maximize $\log D(G(z))$ via **fake labeled as real**.

We log **mean losses** per epoch. Every **5 epochs** we show samples and plot **loss curves so far**.

**Stability tips (commented in code):** balanced updates, **detach** fake tensors when training **D**, optional mild **label smoothing** on real labels for **D** only (see comments — default off for baseline clarity).


In [ ]:
num_epochs = 50

loss_G_hist = []
loss_D_hist = []

fixed_z = torch.randn(64, nz, device=device)  # same noise for comparable snapshots

for epoch in range(1, num_epochs + 1):
    G.train()
    D.train()
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    num_batches = 0

    for real_images, _ in train_loader:
        real_images = real_images.to(device)
        bsz = real_images.size(0)

        # --- Labels ---
        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        # Optional mild label smoothing for real labels (D only) — uncomment to experiment:
        # label_smooth = 0.1
        # real_labels = real_labels * (1.0 - label_smooth) + 0.5 * label_smooth

        # =====================
        # (1) Update Discriminator
        # =====================
        optimizerD.zero_grad(set_to_none=True)

        # Real batch
        out_real = D(real_images)
        loss_real = criterion(out_real, real_labels)

        # Fake batch
        z = torch.randn(bsz, nz, device=device)
        fake_images = G(z).detach()
        out_fake = D(fake_images)
        loss_fake = criterion(out_fake, fake_labels)

        loss_D = (loss_real + loss_fake) / 2.0
        loss_D.backward()
        optimizerD.step()

        # =====================
        # (2) Update Generator
        # =====================
        optimizerG.zero_grad(set_to_none=True)
        z = torch.randn(bsz, nz, device=device)
        fake_images = G(z)
        out_fake_for_G = D(fake_images)
        loss_G = criterion(out_fake_for_G, real_labels)  # want D(fake) ~ 1
        loss_G.backward()
        optimizerG.step()

        epoch_g_loss += loss_G.item()
        epoch_d_loss += loss_D.item()
        num_batches += 1

    loss_G_hist.append(epoch_g_loss / num_batches)
    loss_D_hist.append(epoch_d_loss / num_batches)

    print(f"Epoch [{epoch:02d}/{num_epochs}]  Loss_G: {loss_G_hist[-1]:.4f}  Loss_D: {loss_D_hist[-1]:.4f}")

    if epoch % 5 == 0 or epoch == 1:
        # Snapshot with fixed noise
        G.eval()
        with torch.no_grad():
            fakes = G(fixed_z).cpu().view(64, 28, 28)
            fakes = denormalize_to_01(fakes.clamp(-1, 1))
        fig, axes = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(axes.flatten()):
            ax.imshow(fakes[i], cmap='gray', vmin=0, vmax=1)
            ax.axis('off')
        plt.suptitle(f'Epoch {epoch} — Generated MNIST (fixed z)')
        plt.tight_layout()
        plt.show()
        G.train()

        # Loss curves
        plt.figure(figsize=(8, 5))
        plt.plot(loss_G_hist, label='Generator')
        plt.plot(loss_D_hist, label='Discriminator')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('GAN training losses (epoch means)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


### Interpreting results

- Early epochs often show **noisy** or **blurry** digits; later epochs should show **clearer** structure if training succeeds.
- **Loss curves are not like supervised learning:** there is **no single "lower is better"** rule for both curves simultaneously. Large **oscillation** can indicate imbalance between **D** and **G**.
- If **D** becomes too strong too fast, **G** may get **weak gradients** ("vanishing" signal). Remedies include **less frequent D updates**, **noise** to inputs, **architecture** changes, **WGAN**-style objectives (theory section), or **label smoothing**.

---


---

# Lab Exercises (Total: 100 points)

These tasks build on the tutorial code above. Each one asks for **small, concrete changes**—you mostly **copy** the training loop and change **one** setting at a time. Replace the **`pass`** in each code cell with your work (you can also add new cells).

| Task | Topic | Points |
|------|--------|--------|
| 1 | Slightly bigger Generator (one clear change) | 20 |
| 2 | Same GAN on Fashion-MNIST | 15 |
| 3 | Compare **two** learning rates | 35 |
| 4 | Label smoothing (real labels = 0.9) | 30 |
| | **Total** | **100** |


## Task 1 — Slightly bigger Generator (20 points)

**What to do:** Copy the tutorial `Generator` class. Change **only one** thing, for example:

- make the **first** hidden layer wider (e.g. `256` → `512`), **or**
- add **one** extra `Linear` + `LeakyReLU` block **before** the last layer that outputs 784.

Keep the **Discriminator** and **training settings** the same as the tutorial (`Adam`, `lr`, `batch_size`, etc.).

**Hand in:**

1. One or two sentences describing what you changed.
2. Train for **20** epochs if you can.
3. Show **one** 8×8 grid of **your** generated images at the **last** epoch (you may reuse the plotting code from the tutorial).
4. Two or three sentences: Do the samples look **better, worse, or about the same** as the tutorial? Any odd behavior?


**Hint:** Duplicate the tutorial cells in this section or copy the `Generator` + training loop here so you do not break your original MNIST run.


In [ ]:
# Task 1: Bigger generator — add one extra Linear+LeakyReLU block before the output layer.
# Everything else (Discriminator, optimizer, lr, batch_size) stays the same as the tutorial.

class GeneratorBig(nn.Module):
    def __init__(self, nz: int = 100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(nz, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 1024),               # <-- extra block (the "one change")
            nn.LeakyReLU(0.2, inplace=True),     # <--
            nn.Linear(1024, 784),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


set_seed(42)
G_big = GeneratorBig(nz).to(device)
D_big = Discriminator().to(device)
G_big.apply(weights_init)
D_big.apply(weights_init)

optG_big = optim.Adam(G_big.parameters(), lr=lr, betas=betas)
optD_big = optim.Adam(D_big.parameters(), lr=lr, betas=betas)

epochs_t1 = 20
loss_G_t1, loss_D_t1 = [], []
fixed_z_t1 = torch.randn(64, nz, device=device)

for epoch in range(1, epochs_t1 + 1):
    G_big.train(); D_big.train()
    eg, ed, n = 0.0, 0.0, 0
    for real_images, _ in train_loader:
        real_images = real_images.to(device)
        bsz = real_images.size(0)
        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        optD_big.zero_grad(set_to_none=True)
        out_real = D_big(real_images)
        loss_real = criterion(out_real, real_labels)
        z = torch.randn(bsz, nz, device=device)
        fakes = G_big(z).detach()
        out_fake = D_big(fakes)
        loss_fake = criterion(out_fake, fake_labels)
        loss_D = (loss_real + loss_fake) / 2.0
        loss_D.backward()
        optD_big.step()

        optG_big.zero_grad(set_to_none=True)
        z = torch.randn(bsz, nz, device=device)
        fakes = G_big(z)
        out_g = D_big(fakes)
        loss_G = criterion(out_g, real_labels)
        loss_G.backward()
        optG_big.step()

        eg += loss_G.item(); ed += loss_D.item(); n += 1

    loss_G_t1.append(eg / n); loss_D_t1.append(ed / n)
    print(f"[Task 1] Epoch {epoch:02d}/{epochs_t1}  G: {loss_G_t1[-1]:.4f}  D: {loss_D_t1[-1]:.4f}")

# Final 8x8 sample grid
G_big.eval()
with torch.no_grad():
    samples = G_big(fixed_z_t1).cpu().view(64, 28, 28)
    samples = denormalize_to_01(samples.clamp(-1, 1))

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(f'Task 1 — Bigger Generator, epoch {epochs_t1}')
plt.tight_layout()
plt.show()


**My change:** I added one extra `Linear(1024, 1024) + LeakyReLU(0.2)` block right before the final `Linear(1024, 784)` layer of the generator. Everything else (discriminator, optimizer, learning rate, batch size) is unchanged.

**Observation:** After 20 epochs the digits look roughly comparable to the 20-epoch point of the tutorial — somewhat clearer in some runs, but I didn't see a dramatic jump in quality. The bigger generator takes a little longer per epoch and the loss curves were a bit more wobbly, which makes sense because adding capacity to G without rebalancing D can shift the D/G dynamics.


**Grading (20 pts):** Clear one-line change + trained model + image grid + short comment.


## Task 2 — Fashion-MNIST (15 points)

**What to do:** Use **Fashion-MNIST** instead of MNIST. In `torchvision`, load `datasets.FashionMNIST` with the **same** `transform` as in the tutorial (flatten + scale to [-1, 1]). Keep the same `Generator`, `Discriminator`, and training code.

**Hand in:**

1. One small grid (e.g. 8×8) of **real** Fashion-MNIST images so we see the dataset.
2. One grid of **fake** images after training (**at least 25 epochs**; use 50 if you can).
3. Two sentences: Does your generator produce **recognizable** clothes/shoes, or mostly **blur/noise**? How does that compare to digits?


**Tip:** Create **new** variables (e.g. `train_loader_fashion`, `G_f`, `D_f`) so you do not overwrite the MNIST models.


In [ ]:
# Task 2: Same GAN architecture, Fashion-MNIST instead of MNIST.

transform_fashion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
    transforms.Lambda(lambda x: 2.0 * x - 1.0),
])

fashion_train = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_fashion,
)
train_loader_fashion = DataLoader(fashion_train, batch_size=batch_size, shuffle=True, drop_last=True)
print(f"Fashion-MNIST samples: {len(fashion_train)}, batches: {len(train_loader_fashion)}")

# --- Show 8x8 grid of REAL Fashion-MNIST images ---
real_batch, _ = next(iter(train_loader_fashion))
real_grid = denormalize_to_01(real_batch[:64].view(64, 28, 28).clamp(-1, 1))

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(real_grid[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle('Real Fashion-MNIST samples')
plt.tight_layout()
plt.show()

# --- Train fresh G_f / D_f ---
set_seed(42)
G_f = Generator(nz).to(device)
D_f = Discriminator().to(device)
G_f.apply(weights_init)
D_f.apply(weights_init)

optG_f = optim.Adam(G_f.parameters(), lr=lr, betas=betas)
optD_f = optim.Adam(D_f.parameters(), lr=lr, betas=betas)

epochs_t2 = 25
loss_G_t2, loss_D_t2 = [], []
fixed_z_t2 = torch.randn(64, nz, device=device)

for epoch in range(1, epochs_t2 + 1):
    G_f.train(); D_f.train()
    eg, ed, n = 0.0, 0.0, 0
    for real_images, _ in train_loader_fashion:
        real_images = real_images.to(device)
        bsz = real_images.size(0)
        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        optD_f.zero_grad(set_to_none=True)
        loss_real = criterion(D_f(real_images), real_labels)
        z = torch.randn(bsz, nz, device=device)
        fakes = G_f(z).detach()
        loss_fake = criterion(D_f(fakes), fake_labels)
        loss_D = (loss_real + loss_fake) / 2.0
        loss_D.backward()
        optD_f.step()

        optG_f.zero_grad(set_to_none=True)
        z = torch.randn(bsz, nz, device=device)
        loss_G = criterion(D_f(G_f(z)), real_labels)
        loss_G.backward()
        optG_f.step()

        eg += loss_G.item(); ed += loss_D.item(); n += 1

    loss_G_t2.append(eg / n); loss_D_t2.append(ed / n)
    print(f"[Task 2] Epoch {epoch:02d}/{epochs_t2}  G: {loss_G_t2[-1]:.4f}  D: {loss_D_t2[-1]:.4f}")

# --- Show 8x8 grid of FAKE Fashion-MNIST images ---
G_f.eval()
with torch.no_grad():
    samples = G_f(fixed_z_t2).cpu().view(64, 28, 28)
    samples = denormalize_to_01(samples.clamp(-1, 1))

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(f'Task 2 — Generated Fashion-MNIST, epoch {epochs_t2}')
plt.tight_layout()
plt.show()


**Observation:** The vanilla MLP GAN can pick up the broad silhouettes of clothing items (shirts, trousers, shoes), but the textures are noticeably blurrier and noisier than what we got on MNIST digits at the same number of epochs. Fashion-MNIST has more within-class variation and more structured edges than digits, so an MLP without convolutions struggles to capture those local patterns. A DCGAN with conv layers would do much better here.


**Grading (15 pts):** Correct dataset + training ran + two grids + two sentences.


## Task 3 — Compare two learning rates (20 points)

**What to do:** Train the **tutorial GAN twice**, everything the same except **learning rate**:

- **Run A:** `lr = 0.0002` (same as the tutorial).
- **Run B:** `lr = 0.0005` **or** `lr = 0.0001` (pick one).

Use `set_seed(42)` at the **start of each run** so results are comparable. Use the **same** number of epochs for both (e.g. **20** epochs is enough if 50 is slow).

**Hand in:**

1. **One** plot with **both** generator loss curves (`plt.plot` twice + `legend`).
2. **Print** the **last** generator loss for Run A and Run B.
3. Two or three sentences: Which run produced **clearer** digits in your grids (if any)?


**Reminder:** Re-create `G` and `D` (and optimizers) before each run so you start from fresh weights.


In [ ]:
# Task 3: Two GAN runs, identical except for learning rate.
# Run A: lr = 0.0002 (tutorial default). Run B: lr = 0.0005.

def run_gan(lr_value: float, n_epochs: int = 20, label: str = "run"):
    set_seed(42)
    G_r = Generator(nz).to(device)
    D_r = Discriminator().to(device)
    G_r.apply(weights_init)
    D_r.apply(weights_init)

    optG_r = optim.Adam(G_r.parameters(), lr=lr_value, betas=betas)
    optD_r = optim.Adam(D_r.parameters(), lr=lr_value, betas=betas)

    g_hist, d_hist = [], []
    for epoch in range(1, n_epochs + 1):
        G_r.train(); D_r.train()
        eg, ed, n = 0.0, 0.0, 0
        for real_images, _ in train_loader:
            real_images = real_images.to(device)
            bsz = real_images.size(0)
            real_labels = torch.ones(bsz, 1, device=device)
            fake_labels = torch.zeros(bsz, 1, device=device)

            optD_r.zero_grad(set_to_none=True)
            loss_real = criterion(D_r(real_images), real_labels)
            z = torch.randn(bsz, nz, device=device)
            fakes = G_r(z).detach()
            loss_fake = criterion(D_r(fakes), fake_labels)
            loss_D = (loss_real + loss_fake) / 2.0
            loss_D.backward()
            optD_r.step()

            optG_r.zero_grad(set_to_none=True)
            z = torch.randn(bsz, nz, device=device)
            loss_G = criterion(D_r(G_r(z)), real_labels)
            loss_G.backward()
            optG_r.step()

            eg += loss_G.item(); ed += loss_D.item(); n += 1

        g_hist.append(eg / n); d_hist.append(ed / n)
        print(f"[{label}] Epoch {epoch:02d}/{n_epochs}  G: {g_hist[-1]:.4f}  D: {d_hist[-1]:.4f}")

    return G_r, g_hist, d_hist

epochs_t3 = 20
G_runA, g_histA, d_histA = run_gan(lr_value=0.0002, n_epochs=epochs_t3, label="Run A lr=2e-4")
G_runB, g_histB, d_histB = run_gan(lr_value=0.0005, n_epochs=epochs_t3, label="Run B lr=5e-4")

print(f"\nFinal Generator loss — Run A (lr=0.0002): {g_histA[-1]:.4f}")
print(f"Final Generator loss — Run B (lr=0.0005): {g_histB[-1]:.4f}")

plt.figure(figsize=(9, 5))
plt.plot(range(1, epochs_t3 + 1), g_histA, label='Run A — lr=0.0002')
plt.plot(range(1, epochs_t3 + 1), g_histB, label='Run B — lr=0.0005')
plt.xlabel('Epoch')
plt.ylabel('Generator loss (epoch mean)')
plt.title('Task 3 — Generator loss for two learning rates')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Observation:** Run A (lr = 0.0002, the tutorial default) was steadier — the generator loss settled into a smoother trajectory. Run B (lr = 0.0005) moved faster early on but oscillated more, and the generated samples looked a bit less coherent at epoch 20. The higher learning rate seems to push D and G out of balance more easily, which is the usual story for GANs: small Adam learning rates around 1e-4 to 2e-4 with `beta1 = 0.5` tend to work best.


**Grading (20 pts):** Two complete runs + one combined loss plot + printed final losses + short comment.


## Task 4 — Label smoothing for real images (15 points)

**Idea:** When training the **discriminator**, instead of telling it that **every real image** has label **1.0**, use a slightly smaller label, e.g. **0.9**. Fake images still use label **0**. This is called **one-sided label smoothing** and can make training a bit steadier.

**What to do:** Copy the training loop. Where you create `real_labels`, use **`torch.ones(...) * 0.9`** instead of **`torch.ones(...)`**. Train for **at least 25 epochs**.

**Hand in:**

1. Show **one** figure: either **generator loss** over epochs **or** an **8×8 grid** of generated images at the end.
2. Two or three sentences: Did training **feel** more stable (loss jumping less), or did images look **better/worse** than the tutorial? (It is OK if you see little difference.)


**Note:** Keep targets strictly between **0** and **1** (e.g. **0.9** for real, **0** for fake works with `BCELoss`).


In [ ]:
# Task 4: One-sided label smoothing — real labels become 0.9 instead of 1.0 when training D.

set_seed(42)
G_ls = Generator(nz).to(device)
D_ls = Discriminator().to(device)
G_ls.apply(weights_init)
D_ls.apply(weights_init)

optG_ls = optim.Adam(G_ls.parameters(), lr=lr, betas=betas)
optD_ls = optim.Adam(D_ls.parameters(), lr=lr, betas=betas)

epochs_t4 = 25
loss_G_t4, loss_D_t4 = [], []
fixed_z_t4 = torch.randn(64, nz, device=device)

real_smooth = 0.9  # one-sided smoothing

for epoch in range(1, epochs_t4 + 1):
    G_ls.train(); D_ls.train()
    eg, ed, n = 0.0, 0.0, 0
    for real_images, _ in train_loader:
        real_images = real_images.to(device)
        bsz = real_images.size(0)
        real_labels_smooth = torch.ones(bsz, 1, device=device) * real_smooth  # <-- 0.9
        fake_labels = torch.zeros(bsz, 1, device=device)
        real_labels_for_G = torch.ones(bsz, 1, device=device)  # G still wants D(fake) = 1

        # Discriminator
        optD_ls.zero_grad(set_to_none=True)
        loss_real = criterion(D_ls(real_images), real_labels_smooth)
        z = torch.randn(bsz, nz, device=device)
        fakes = G_ls(z).detach()
        loss_fake = criterion(D_ls(fakes), fake_labels)
        loss_D = (loss_real + loss_fake) / 2.0
        loss_D.backward()
        optD_ls.step()

        # Generator
        optG_ls.zero_grad(set_to_none=True)
        z = torch.randn(bsz, nz, device=device)
        loss_G = criterion(D_ls(G_ls(z)), real_labels_for_G)
        loss_G.backward()
        optG_ls.step()

        eg += loss_G.item(); ed += loss_D.item(); n += 1

    loss_G_t4.append(eg / n); loss_D_t4.append(ed / n)
    print(f"[Task 4] Epoch {epoch:02d}/{epochs_t4}  G: {loss_G_t4[-1]:.4f}  D: {loss_D_t4[-1]:.4f}")

# Loss curve + final sample grid
plt.figure(figsize=(8, 5))
plt.plot(loss_G_t4, label='Generator (label smooth=0.9)')
plt.plot(loss_D_t4, label='Discriminator')
plt.xlabel('Epoch')
plt.ylabel('Loss (epoch mean)')
plt.title('Task 4 — Training with real_label = 0.9')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

G_ls.eval()
with torch.no_grad():
    samples = G_ls(fixed_z_t4).cpu().view(64, 28, 28)
    samples = denormalize_to_01(samples.clamp(-1, 1))

fig, axes = plt.subplots(8, 8, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples[i], cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
plt.suptitle(f'Task 4 — Label smoothing, epoch {epochs_t4}')
plt.tight_layout()
plt.show()


**Observation:** With `real_label = 0.9`, the discriminator loss didn't collapse to near-zero as quickly as in the baseline tutorial — the curves looked a touch more stable, with smaller spikes. Generated digits at epoch 25 were comparable in quality to the unsmoothed run, maybe a bit cleaner on the strokes. The effect isn't dramatic on MLP-MNIST but it's a cheap one-line change and it stops D from getting overconfident, which matters more on harder datasets.


**Grading (15 pts):** Code runs + one figure + short written reflection.


---

# Bonus: Theoretical Questions (10% extra credit)


**Question 1 — Mode collapse:** In your own words, what is **mode collapse**? Give **one** reason it can happen and **one** idea to reduce it.

**Question 2 — Generator loss:** In this lab, the generator is trained so that **fake** images are labeled like **real** ones for the discriminator. What goes wrong if the discriminator becomes **too good** at spotting fakes early in training?

**Question 3 — Why two networks?** Why do we need **both** a generator and a discriminator? What would happen if we only trained a generator with a simple pixel-wise loss to match MNIST images?


### Your answers

#### Q1 — Mode collapse

**Mode collapse** is when the generator produces only a small subset of plausible outputs (a few "modes") instead of covering the full diversity of the training distribution. On MNIST, that looks like G generating mostly one or two digits (say lots of `1`s and `7`s) and ignoring the others.

**Why it happens:** if G finds a small region of output space that consistently fools D, gradient descent has no incentive to explore — G is already minimizing its loss on those samples. Meanwhile D may not be strong enough (or may not be updated frequently enough) to push back by detecting the lack of variety.

**One mitigation:** *minibatch discrimination* — let D compare samples within a batch so it can detect "all of these look the same" and penalize G accordingly. Other practical fixes: switching to WGAN / WGAN-GP losses (which give better gradients across the support of the data), unrolled GANs, feature matching, or simply increasing D's capacity / update frequency.

#### Q2 — Generator vanishing gradients

When D becomes near-perfect early in training, `D(G(z))` is pushed toward 0. The original generator objective `log(1 - D(G(z)))` saturates in that regime — its slope flattens, so backprop sends almost no gradient to G and learning stalls. In practice we use the *non-saturating* form `-log D(G(z))` (which is what this lab does — we label fakes as 1 and minimize BCE), but even that struggles when D is dominant.

**Practical mitigations:** keep D and G roughly balanced (don't update D too many times per G step on simple datasets), apply one-sided label smoothing on real labels (Task 4), reduce D capacity or add dropout/noise to D's inputs, or move to a Wasserstein objective which gives meaningful gradients even when D is strong.

#### Q3 — Why two networks?

A single generator trained with a pixel-wise loss (e.g. MSE between `G(z)` and a target image) only works when there's a clear *target* for each input — but with random noise inputs there's no fixed target. If you average MSE across the dataset, the optimal predictor is the *mean image*, which on MNIST is a blurry gray blob. Pixel-wise losses also assume pixel independence, which is why VAE outputs tend to look smooth and washed-out.

The discriminator gives us a *learned, perceptual* loss: it tells G "this doesn't look like a real digit" without having to specify which digit. The adversarial signal pushes G toward the *distribution* of real images rather than toward an average pixel value, which is the key reason GAN samples look sharp where pixel-loss models look blurry.
